# Disproof of the Blossom-to-Bridge Lemma

This notebook gives a fully reproducible counterexample to the proposed lemma:

> Let \(R\) be a connected \(\{1,3\}\)-graph with at least three leaves and let
> \(N\) be a matching such that \(R-N\) is connected and unicyclic. If \(R\)
> is not matching-decyclable, then \(R\) has a bridge with a leafless side.

The statement is false.

We verify an explicit 16-vertex counterexample, prove its failure through a
finite Hamilton-path certificate, and turn it into a bridgeless cubic graph
with a fixed-cycle one-defect decomposition that cannot be repaired while
preserving the chosen cycle.

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("networkx") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "networkx>=3.2"]
    )

from collections import Counter
import itertools
import networkx as nx

print("NetworkX", nx.__version__)

NetworkX 3.6.1


## 1. The counterexample

The graph is encoded by the graph6 string

```text
O??DQ?[C?AaAA?@O??wGS
```

Its leaves are \(0,1,2\); every other vertex has degree three.

In [2]:
graph6 = b"O??DQ?[C?AaAA?@O??wGS"
R = nx.from_graph6_bytes(graph6)

print("vertices:", R.number_of_nodes())
print("edges:", R.number_of_edges())
print("leaves:", [v for v, d in R.degree() if d == 1])
print("degree counts:", Counter(dict(R.degree()).values()))
print("edge list:")
for e in sorted(tuple(sorted(e)) for e in R.edges()):
    print(e)

assert R.number_of_nodes() == 16
assert R.number_of_edges() == 21
assert Counter(dict(R.degree()).values()) == Counter({3: 13, 1: 3})

vertices: 16
edges: 21
leaves: [0, 1, 2]
degree counts: Counter({3: 13, 1: 3})
edge list:
(0, 6)
(1, 7)
(2, 6)
(3, 8)
(3, 9)
(3, 11)
(4, 6)
(4, 8)
(4, 12)
(5, 8)
(5, 13)
(5, 15)
(7, 10)
(7, 13)
(9, 10)
(9, 11)
(10, 15)
(11, 14)
(12, 14)
(12, 15)
(13, 14)


## 2. There is no bridge with a leafless side

For each bridge, both components of its deletion contain an original leaf.

In [3]:
leaves = {v for v, d in R.degree() if d == 1}
bridges = list(nx.bridges(R))

bridge_table = []
for u, v in bridges:
    H = R.copy()
    H.remove_edge(u, v)
    sides = [set(c) for c in nx.connected_components(H)]
    leaf_sets = [sorted(side & leaves) for side in sides]
    bridge_table.append(((u, v), leaf_sets))
    assert all(side & leaves for side in sides)

print("bridges and leaves on their two sides:")
for row in bridge_table:
    print(row)

bridges and leaves on their two sides:
((0, 6), [[0], [1, 2]])
((1, 7), [[0, 2], [1]])
((2, 6), [[0, 1], [2]])
((4, 6), [[0, 2], [1]])


## 3. The graph is one matching edge away from a tree complement

The matching

\[
N_0=\{38,4\,12,5\,13,7\,10,9\,11\}
\]

has five edges and \(R-N_0\) is connected and unicyclic.

In [4]:
N0 = {
    (3, 8),
    (4, 12),
    (5, 13),
    (7, 10),
    (9, 11),
}

U = R.copy()
U.remove_edges_from(N0)

cycle_rank = (
    U.number_of_edges()
    - U.number_of_nodes()
    + nx.number_connected_components(U)
)

print("matching N0:", sorted(N0))
print("connected:", nx.is_connected(U))
print("cycle rank:", cycle_rank)
print("unique cycle:", nx.cycle_basis(U))

assert nx.is_matching(R, N0)
assert nx.is_connected(U)
assert cycle_rank == 1

matching N0: [(3, 8), (4, 12), (5, 13), (7, 10), (9, 11)]
connected: True
cycle rank: 1
unique cycle: [[10, 9, 3, 11, 14, 12, 15]]


## 4. Exhaustive matching certificate

Since \(R\) has \(21\) edges and \(16\) vertices, a matching whose complement
is a spanning tree must have

\[
21-(16-1)=6
\]

edges.

The code enumerates all matchings of sizes five and six.

In [5]:
edges = [tuple(sorted(e)) for e in R.edges()]

def is_matching(edge_set):
    used = set()
    for u, v in edge_set:
        if u in used or v in used:
            return False
        used.add(u)
        used.add(v)
    return True

results = {}
for size in (5, 6):
    number_of_matchings = 0
    connected_complements = 0
    connected_ranks = Counter()

    for candidate in itertools.combinations(edges, size):
        if not is_matching(candidate):
            continue

        number_of_matchings += 1
        complement = R.copy()
        complement.remove_edges_from(candidate)

        if nx.is_connected(complement):
            connected_complements += 1
            rank = (
                complement.number_of_edges()
                - complement.number_of_nodes()
                + 1
            )
            connected_ranks[rank] += 1

    results[size] = (
        number_of_matchings,
        connected_complements,
        connected_ranks,
    )

for size, result in results.items():
    print(f"size {size}: {result}")

assert results[5][0] == 1448
assert results[5][1] == 105
assert results[5][2] == Counter({1: 105})

assert results[6][0] == 615
assert results[6][1] == 0

size 5: (1448, 105, Counter({1: 105}))
size 6: (615, 0, Counter())


Therefore \(R\) satisfies all hypotheses of the proposed lemma but is not
matching-decyclable.

## 5. A compact human-checkable obstruction

In a connected subcubic graph, matching-decyclability is equivalent to the
existence of a spanning tree whose leaves have original degree at most two.

Here the only possible tree leaves are \(0,1,2\). Every bridge belongs to
every spanning tree. In particular, \(06,26,46\) force vertex \(6\) to have
tree degree three.

A subcubic tree with three leaves has exactly one degree-three vertex.
Hence vertex \(6\) would be the unique branch vertex. Deleting
\(0,1,2,6\) would leave a Hamiltonian path from \(4\) to \(7\) in the
12-vertex graph below.

The degree equations for a spanning \(4\)-\(7\) path have exactly five
solutions. Each consists of a path plus a disjoint cycle.

In [6]:
core_vertices = set(R) - {0, 1, 2, 6}
H = R.subgraph(core_vertices).copy()
core_edges = [tuple(sorted(e)) for e in H.edges()]

candidates = []
for chosen in itertools.combinations(core_edges, H.number_of_nodes() - 1):
    degree = Counter()
    for u, v in chosen:
        degree[u] += 1
        degree[v] += 1

    correct_degrees = all(
        degree[v] == (1 if v in {4, 7} else 2)
        for v in H
    )
    if not correct_degrees:
        continue

    K = nx.Graph()
    K.add_nodes_from(H)
    K.add_edges_from(chosen)
    candidates.append(K)

print("degree-feasible candidates:", len(candidates))
print("connected candidates:", sum(nx.is_connected(K) for K in candidates))

assert len(candidates) == 5
assert all(not nx.is_connected(K) for K in candidates)

def describe_component(C):
    degrees = dict(C.degree())
    if all(d == 2 for d in degrees.values()):
        return "cycle", nx.cycle_basis(C)[0]

    endpoints = [v for v, d in degrees.items() if d == 1]
    return "path", nx.shortest_path(C, endpoints[0], endpoints[1])

for index, K in enumerate(candidates, start=1):
    description = []
    for component_vertices in nx.connected_components(K):
        C = K.subgraph(component_vertices).copy()
        description.append(describe_component(C))
    print(index, description)

degree-feasible candidates: 5
connected candidates: 0
1 [('cycle', [11, 9, 3, 8, 5, 13, 14]), ('path', [4, 12, 15, 10, 7])]
2 [('path', [4, 8, 3, 11, 9, 10, 7]), ('cycle', [5, 13, 14, 12, 15])]
3 [('cycle', [5, 8, 3, 11, 9, 10, 15]), ('path', [4, 12, 14, 13, 7])]
4 [('cycle', [3, 9, 11]), ('path', [4, 8, 5, 13, 14, 12, 15, 10, 7])]
5 [('cycle', [10, 9, 3, 11, 14, 12, 15]), ('path', [4, 8, 5, 13, 7])]


The five candidates are:

1. cycle \(11,9,3,8,5,13,14\) and path \(4,12,15,10,7\);
2. path \(4,8,3,11,9,10,7\) and cycle \(5,13,14,12,15\);
3. cycle \(5,8,3,11,9,10,15\) and path \(4,12,14,13,7\);
4. cycle \(3,9,11\) and path \(4,8,5,13,14,12,15,10,7\);
5. cycle \(10,9,3,11,14,12,15\) and path \(4,8,5,13,7\).

Thus no Hamiltonian \(4\)-\(7\) path exists and no matching can leave a tree.

## 6. A bridgeless cubic fixed-cycle counterexample

Add the triangle \(01,12,20\) on the three leaves. The resulting graph \(G\)
is cubic and bridgeless.

Taking the triangle as the fixed cycle component, the matching \(N_0\)
above and one edge of the unique cycle of \(R-N_0\) yield a one-defect
decomposition. It cannot be repaired while retaining that triangle,
because this would require a decycling matching of \(R\).

In [7]:
G = R.copy()
triangle = {(0, 1), (1, 2), (0, 2)}
G.add_edges_from(triangle)

assert set(dict(G.degree()).values()) == {3}
assert list(nx.bridges(G)) == []

# Split N0 into a harmless matching and one passive defect edge.
passive = (7, 10)
cycle_edge = (10, 15)
M = N0 - {passive}
P2 = {passive, cycle_edge}

tree_edges = {
    tuple(sorted(e))
    for e in U.edges()
    if tuple(sorted(e)) != tuple(sorted(cycle_edge))
}

T = nx.Graph()
T.add_nodes_from(G)
T.add_edges_from(tree_edges)

partition = tree_edges | M | P2 | triangle
all_edges = {tuple(sorted(e)) for e in G.edges()}

print("cubic graph6:", nx.to_graph6_bytes(G, header=False).strip())
print("tree edges:", sorted(tree_edges))
print("fixed triangle:", sorted(triangle))
print("matching:", sorted(M))
print("P2 defect:", sorted(P2))

assert nx.is_tree(T)
assert nx.is_matching(G, M)
assert partition == all_edges
assert sum(map(len, [tree_edges, M, P2, triangle])) == len(all_edges)

cubic graph6: b'Ow?DQ?[C?AaAA?@O??wGS'
tree edges: [(0, 6), (1, 7), (2, 6), (3, 9), (3, 11), (4, 6), (4, 8), (5, 8), (5, 15), (7, 13), (9, 10), (11, 14), (12, 14), (12, 15), (13, 14)]
fixed triangle: [(0, 1), (0, 2), (1, 2)]
matching: [(3, 8), (4, 12), (5, 13), (9, 11)]
P2 defect: [(7, 10), (10, 15)]


## 7. Consequence

The counterexample disproves:

* the Blossom-to-Bridge Lemma;
* the claimed fixed-cycle repair theorem for bridgeless cubic graphs.

It does **not** disprove the unrestricted one-defect theorem, because a
successful repair may alter or destroy the chosen cycle component.

The accompanying note proves a stronger complexity result: deciding whether
such a near-unicyclic matching can be completed remains NP-complete even for
\(\{1,3\}\)-graphs with exactly three leaves and no leafless bridge side.